# Côte d'Ivoire - PROMIRE project / UN-REDD

3 · Forest emissions and removals 2021-2025 - activity data and calculation workbook
-------
This notebook prepares the activity data at the level of detail required to compute emissions and removals, then produces a **dynamic** Excel workbook (all result cells are formulas):

| Component | Activity data | Factor |
|:--|:--|:--|
| Deforestation - biomass and dead organic matter | Def: zone × forest 2021 × land use 2025 | net EF = forest stock − post-deforestation stock |
| Deforestation - mineral soils | same | IPCC eq. 2.25: SOC_ref · (F − 1) over D = 20 years |
| Deforestation - biomass burning (CH₄, N₂O) | same, × burnt fraction | IPCC eq. 2.27: MB · Cf · Gef · 10⁻³ × GWP |
| Degradation | Deg: zone × forest 2021 × forest 2025 | stock of forest 2021 − stock of forest 2025 |
| Enhancement of stocks (gains) | Gain: zone × gain category | annual growth rate |

**Calculation chain**

1. **Activity** of each sample: output of notebook `1`.
2. **Categories** of each sample: forest category in 2021 and in 2025, post-deforestation land use in 2025, gain category.
3. **Areas**: stratified estimator by zone (zone × stratum post-stratification, as in notebook `2`); each transition is a domain of the zone.
4. **Factors**: carbon stocks, fuel biomass, soil factors, burning parameters and growth rates (national values of the FREL 2024 and IPCC default values), recomputed with formulas.
5. **Emissions, removals and uncertainties**: propagation with IPCC approach 1; soils and gains accumulate their effects from one year to the next (cohorts).

**Inputs**: `results/CIV_activity_samples_2021_2025_en.csv` (notebook `1`) and the phytogeographic zones (`data/ZnePhytoGeo_CI.shp`).

**Outputs** (`results/` folder), workbooks produced by the module `sbae_emissions_excel_en.py`:

- `CIV_emissions_deforestation_2021_2025_en.xlsx`: national and by zone;
- `CIV_emissions_deforestation_2021_2025_ERP_en.xlsx`: territory of the ERP (Emission Reductions Program, *PRE* in French), section 12.

The checks of section 10 compare the workbook values with independent calculations.

*English version of `3_CIV_emissions_2021_2025.ipynb`: same calculations, same results. Abbreviations: EF emission factor, DOM dead organic matter, DW deadwood, MB fuel biomass, t d.m. tonnes of dry matter, GWP global warming potential, NFI national forest inventory (IFN), FREL forest reference emission level (NERF).*

In [ ]:
import os
from statistics import NormalDist
import numpy as np
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Parameters

In [ ]:
data_path  = 'results/CIV_activity_samples_2021_2025_en.csv'   # output of notebook 1
zones_path = 'data/ZnePhytoGeo_CI.shp'
xlsx_path  = 'results/CIV_emissions_deforestation_2021_2025_en.xlsx'

initial_year, final_year = 2021, 2025      # length of the period T = 4 years
years = list(range(initial_year, final_year + 1))
confidence    = 0.90     # confidence level of the results
ef_confidence = 0.90     # confidence level of the input uncertainties (FREL / NFI: 90% CI)
forest_class  = 'Terres_Forestieres'

# Mineral soils: LAC soils, tropical moist climate
socref   = 47.0          # tC/ha (IPCC table 2.3)
u_socref = 0.0           # uncertainty not available: to be completed
soc_d    = 20            # years

# Samples outside the phytogeographic zones are assigned to the nearest zone (as in notebook 2)
assign_nearest_zone = True

## 2. Samples and activity (output of notebook 1)

In [ ]:
ad = pd.read_csv(data_path)
print(len(ad), 'samples')
ad.activity.value_counts()

In [ ]:
# Phytogeographic zone of each sample (same rule as in notebook 2)
zones = gpd.read_file(zones_path)
gdf = gpd.GeoDataFrame(ad, geometry=gpd.points_from_xy(ad.center_lon, ad.center_lat), crs='EPSG:4326').to_crs('EPSG:32630')
gdf = gpd.sjoin_nearest(gdf, zones[['Zone', 'geometry']].to_crs('EPSG:32630'), how='left', distance_col='zdistance')
assert gdf.plotid.is_unique
ad = pd.DataFrame(gdf.drop(columns=['geometry', 'index_right'])).rename(columns={'Zone': 'zone'})
print("Samples outside the zones:", (ad.zdistance > 0).sum())
if not assign_nearest_zone:
    ad = ad[ad.zdistance == 0]

# Valid samples for the estimator: known stratum and determined activity (as in notebook 2)
valid = ad[ad.strata.notna() & (ad.activity != 'ND')].copy()
valid['strata'] = valid['strata'].astype(int)
print(len(valid), 'valid samples')
valid.activity.value_counts()

## 3. CEO classes of 2021 and 2025

For each sample: level-2 class in 2021 (`class_initial`) and in 2025 (`class_final`). Depending on the activity, they determine:

- **Def**: the forest stock that is lost (2021 class) and the land use that follows (2025 class, always non-forest);
- **Deg**: the 2021 forest and the 2025 forest;
- **Gain**: the type of forest that establishes (2025 class).

In [ ]:
valid['class_initial'] = valid[f'n2_{initial_year}']
valid['class_final'] = valid[f'n2_{final_year}']
is_def = valid.activity == 'Def'
assert (valid.loc[is_def, f'n1_{final_year}'] != forest_class).all()

# Year of the deforestation event (first non-forest year) - information only
valid['def_year'] = valid[is_def].apply(
    lambda r: next((y for y in years[1:] if r[f'n1_{y}'] != forest_class), np.nan), axis=1).reindex(valid.index)

for code in ['Def', 'Deg', 'Gain']:
    d = valid[valid.activity == code]
    print(f'\n{code}: {len(d)} samples (2021 class in rows, 2025 class in columns)')
    print(pd.crosstab(d.class_initial, d.class_final).to_string())

In [ ]:
print("Def samples by year of the event (information only):")
valid.loc[is_def, 'def_year'].astype(int).value_counts().sort_index()

## 4. Mapping of the CEO classes to the categories

The categories are those of the national emission factors (FREL 2024). These tables are reproduced in the "Mappings" sheet of the workbook, where they remain editable.

In [ ]:
forest_cats = ['Undegraded forest', 'Secondary or degraded forest', 'Agroforest']
post_cats = ['Annual crops', 'Cocoa', 'Rubber', 'Oil palm', 'Perennial crops', 'Fallow / thicket',
             'Grassland', 'No biomass']
gain_cats = ['Secondary forest', 'Agroforest', 'Plantation / reforestation']

forest_map = {
    'Foret_Dense': 'Undegraded forest',
    'Foret_Galerie': 'Undegraded forest',
    'Foret_Sol_Hydromorphe': 'Undegraded forest',
    'Mangrove': 'Undegraded forest',
    'Foret_Secondaire_Degradee': 'Secondary or degraded forest',
    'Plantation_Agroforestiere': 'Agroforest',
    'Plantation_Forestiere_Reboisement': 'Secondary or degraded forest',
}
post_map = {
    'Amenagement_Agricole': 'Annual crops',
    'Cacao': 'Cocoa',
    'Hevea': 'Rubber',
    'Palmier': 'Oil palm',
    'Anacarde': 'Perennial crops',
    'Cafe': 'Perennial crops',
    'Cocoteraie': 'Perennial crops',
    'Plantation_Fruitiere': 'Perennial crops',
    'Jachere_Fourree': 'Fallow / thicket',
    'Savane_Arboree': 'Grassland',
    'Savane_Arbustive': 'Grassland',
    'Savane_Herbeuse': 'Grassland',
    'Habitation': 'No biomass',
    'Infrastructures': 'No biomass',
    'Affleurement_Rocheux': 'No biomass',
    'Sol_Nu': 'No biomass',
    'Cours_Voie_Eau': 'No biomass',
    'Lac': 'No biomass',
    'Plan_Eau': 'No biomass',
    'Zones_Marecageuses': 'No biomass',
}
gain_map = {
    'Foret_Dense': 'Secondary forest',
    'Foret_Galerie': 'Secondary forest',
    'Foret_Sol_Hydromorphe': 'Secondary forest',
    'Mangrove': 'Secondary forest',
    'Foret_Secondaire_Degradee': 'Secondary forest',
    'Plantation_Agroforestiere': 'Agroforest',
    'Plantation_Forestiere_Reboisement': 'Plantation / reforestation',
}
mapping_notes = [
    'Gallery forest, forest on hydromorphic soil and mangrove: stock of the dense forest (undegraded forest) of the zone.',
    'Forest plantation / reforestation: stock of the secondary forest by default. For gains, its own growth rate ("Plantation / reforestation").',
    'Fallow / thicket: biomass stock of grassland and soil factors of annual crops ("Thicket" category).',
    'Cashew, coffee, coconut, fruit plantation: stock of perennial crops (mean of rubber / oil palm).',
    'Settlements, infrastructure, bare soil, rock outcrops, water and wetlands: zero biomass stock and total loss of soil carbon (settlements and other land).',
    'Gains: any natural forest that establishes grows at the rate of the secondary forest of the zone.',
]

# Every class used must have a category
act = valid.activity
assert valid.loc[act.isin(['Def', 'Deg']), 'class_initial'].isin(forest_map).all()
assert valid.loc[act == 'Def', 'class_final'].isin(post_map).all()
assert valid.loc[act == 'Deg', 'class_final'].isin(forest_map).all()
assert valid.loc[act == 'Gain', 'class_final'].isin(gain_map).all()
assert set(forest_map.values()) <= set(forest_cats) and set(post_map.values()) <= set(post_cats)
assert set(gain_map.values()) <= set(gain_cats)
print('All the classes of the Def, Deg and Gain samples are assigned.')

## 5. Carbon stocks

National values of the FREL 2024. The uncertainties are half-widths of the 90% CI expressed as fractions (14.58% → 0.1458).

- **AGB, BGB**: national forest inventory (NFI), Tier 2, in tonnes of dry matter per hectare, and their uncertainties.
- **Litter**: IPCC Tier 1 in tC/ha; uncertainty taken from the upper bound of the IPCC range. For secondary forest, litter proportional to AGB relative to dense forest.
- **Deadwood**: NFI, four components and their relative errors.
- **CF** = 0.47; uncertainty = |0.44 − 0.47| / 0.47 = 6.4% (IPCC 2006, table 4.3).

In [ ]:
ZO, ZM, ZS, ZSS = 'Secteur Ombrophile', 'Secteur mésophile', 'Secteur Soudanais', 'Secteur Sub Soudanais'
FND, FSD, AGF = forest_cats


def fstock(cat, zone, agb, u_agb, bgb, u_bgb, dw, e, litter=None, src=''):
    """One forest stock row. dw / e: (lying, stumps, standing, standing roots) in t d.m./ha / fraction."""
    row = dict(category=cat, zone=zone, agb=agb, u_agb=u_agb, bgb=bgb, u_bgb=u_bgb,
               dw_lying=dw[0], dw_stump=dw[1], dw_standing=dw[2], dw_bgb_standing=dw[3],
               e_lying=e[0], e_stump=e[1], e_standing=e[2], e_bgb_standing=e[3], source=src,
               litter_rule='none', litter_tc=np.nan, litter_upper_tc=np.nan, litter_ref=None)
    if litter == 'scaled':
        row.update(litter_rule='scaled', litter_ref=FND)
    elif litter is not None:
        row.update(litter_rule='ipcc', litter_tc=litter[0], litter_upper_tc=litter[1])
    return row


NO_DW = ((0, 0, 0, 0), (0, 0, 0, 0))
forest_stocks = pd.DataFrame([
    # undegraded forest (dense forest)
    fstock(FND, ZO,  204.57, .1458, 75.69, .1458, (16.96, .33, 13.59, 3.80), (.2624, .3162, .2627, .2360), (4.8, 16.4), 'FREL 2024 (NFI)'),
    fstock(FND, ZM,  134.70, .1688, 30.60, .1992, (13.95, .24, 14.36, 3.96), (.2903, .5316, .4321, .3852), (4.8, 16.4), 'FREL 2024 (NFI)'),
    fstock(FND, ZS,  104.30, .0939, 29.21, .0939, (3.01, .03, 5.67, 1.67), (.5082, 1.0007, .2662, .2494), (5.9, 14.8), 'FREL 2024 (NFI)'),
    fstock(FND, ZSS, 101.40, .3614, 22.43, .4334, (8.93, .64, 12.19, 3.57), (.5515, 1.0393, .4252, .3974), (5.9, 14.8), 'FREL 2024 (NFI)'),
    # secondary or degraded forest: litter proportional to AGB
    fstock(FSD, ZO,  107.71, .1441, 39.85, .1441, (17.58, .50, 12.78, 3.64), (.2312, .2666, .2314, .2110), 'scaled', 'FREL 2024 (NFI)'),
    fstock(FSD, ZM,   67.89, .1417, 13.58, .1417, (27.55, .60, 21.11, 5.66), (.3933, .4660, .4026, .3611), 'scaled', 'FREL 2024 (NFI)'),
    fstock(FSD, ZS,   70.71, .2198, 19.80, .2198, (3.99, .10, 11.23, 3.11), (.4417, .9661, .4829, .4488), 'scaled', 'FREL 2024 (NFI)'),
    fstock(FSD, ZSS,  48.11, .2643,  9.62, .2643, (4.88, .14, 6.59, 1.94), (.4172, .9215, .7130, .6569), 'scaled', 'FREL 2024 (NFI)'),
    # agroforest: identical for all zones, without litter or deadwood
    *[fstock(AGF, z, 45.80, .0954, 8.40, .1321, *NO_DW, None, 'FREL 2024 (NFI)') for z in (ZO, ZM, ZS, ZSS)],
])

post_stocks = pd.DataFrame([
    dict(category='Annual crops',     rule='input', agb=5.53,   u_agb=.7505,  bgb=0.0,  u_bgb=0.0,   source='FREL 2024 (agricultural land)'),
    dict(category='Cocoa',            rule='input', agb=37.2,   u_agb=.1282,  bgb=8.2,  u_bgb=.1207, source='FREL 2024'),
    dict(category='Rubber',           rule='input', agb=113.4,  u_agb=.1828,  bgb=22.8, u_bgb=.1803, source='FREL 2024 (rubber 10 years)'),
    dict(category='Oil palm',         rule='input', agb=60.0,   u_agb=.2330,  bgb=12.0, u_bgb=.2333, source='FREL 2024 (oil palm 20 years)'),
    dict(category='Perennial crops',  rule='mean', mean_of=['Rubber', 'Oil palm'], agb=np.nan, u_agb=np.nan,
         bgb=np.nan, u_bgb=np.nan, source='FREL 2024 (mean of rubber / oil palm)'),
    dict(category='Fallow / thicket', rule='mean', mean_of=['Grassland'], agb=np.nan, u_agb=np.nan,
         bgb=np.nan, u_bgb=np.nan, source='Stock of grassland ("Thicket")'),
    dict(category='Grassland',        rule='input', agb=35.33,  u_agb=1.2479, bgb=4.55, u_bgb=1.0593, source='FREL 2024'),
    dict(category='No biomass',       rule='zero',  agb=0.0,    u_agb=0.0,    bgb=0.0,  u_bgb=0.0,   source='Zero stock (settlements, other land)'),
])
forest_stocks[['category', 'zone', 'agb', 'u_agb', 'bgb', 'u_bgb', 'litter_rule', 'litter_tc', 'litter_upper_tc']]

## 6. Mineral soil factors

IPCC equation 2.25: the change in soil stock is SOC_ref · (F − 1) over D = 20 years, with F = F_LU · F_MG · F_I for the land use that follows deforestation (F = 1 for forest). Factors of the FREL 2024. No uncertainty is available for these factors: they are set to 0 and can be completed in the workbook.

In [ ]:
def soc(cat, flu, fmg, fi, src):
    return dict(category=cat, flu=flu, u_flu=0.0, fmg=fmg, u_fmg=0.0, fi=fi, u_fi=0.0, source=src)


soc_factors = pd.DataFrame([
    soc('Annual crops',     0.83, 1.04, 0.92, 'FREL 2024: agricultural land'),
    soc('Cocoa',            1.01, 1.04, 0.92, 'FREL 2024: cocoa'),
    soc('Rubber',           1.01, 1.04, 0.92, 'FREL 2024: rubber'),
    soc('Oil palm',         1.01, 1.04, 0.92, 'Factors of perennial crops'),
    soc('Perennial crops',  1.01, 1.04, 0.92, 'FREL 2024: perennial agriculture, cashew'),
    soc('Fallow / thicket', 0.83, 1.04, 0.92, 'FREL 2024: thicket'),
    soc('Grassland',        1.00, 1.00, 1.00, 'FREL 2024: grassland (no change)'),
    soc('No biomass',       0.00, 0.00, 0.00, 'FREL 2024: settlements, other land (total loss)'),
])
assert list(soc_factors.category) == post_cats
soc_factors

## 6b. Biomass burning during deforestation

IPCC equation 2.27 (2006, vol. 4): L = A_burnt · MB · Cf · Gef · 10⁻³ (t of CH₄ or N₂O), then × GWP (AR5).

- **MB**, fuel biomass in t d.m./ha = AGB + litter + deadwood (without roots), computed in the workbook from the stocks of section 5; litter (tC/ha) is converted to dry matter with the carbon fraction of litter, 0.37.
- **Cf** = 0.32; **Gef** = 6.8 g/kg (CH₄) and 0.2 g/kg (N₂O); **GWP** = 28 (CH₄) and 265 (N₂O).
- **Burnt fraction** of the deforested area: 0.5, except 0 for conversions to fallow / thicket or to grassland and for agroforest converted to cocoa or to perennial crops.
- No uncertainty is available for these parameters: they are set to 0 and can be completed in the workbook. The uncertainties of the areas and of MB are propagated.

In [ ]:
# Burning parameters (IPCC 2006; uncertainties not available: to be completed)
fire = dict(cf_litter=0.37,                     # carbon fraction of litter
            comb=0.32, u_comb=0.0,              # combustion factor Cf
            gef_ch4=6.8, u_gef_ch4=0.0,         # g CH4 / kg d.m. burnt
            gef_n2o=0.2, u_gef_n2o=0.0,         # g N2O / kg d.m. burnt
            gwp_ch4=28, gwp_n2o=265,            # GWP AR5
            u_fire=0.0)                         # uncertainty of the burnt fraction

# Burnt fraction by transition: 0.5 except conversions without fire
no_fire = {'Fallow / thicket', 'Grassland'}
fire_fraction = {(f, u): 0.0 if u in no_fire or (f == AGF and u in ('Cocoa', 'Perennial crops')) else 0.5
                 for f in forest_cats for u in post_cats}
pd.Series(fire_fraction).unstack()

## 7. Growth rates of the gains

Net growth of above-ground biomass, in tC/ha/yr, and uncertainty (90% CI), values of the FREL 2024. Only above-ground growth is counted (R = 0); the workbook allows a root-to-shoot ratio R to be entered. For the zone × forest type combinations without a value of their own, the closest value is used (same forest type, same FAO ecological zone where possible); they are flagged in the source column.

In [ ]:
def growth(cat, zone, rate, u_rate, src):
    return dict(category=cat, zone=zone, rate=rate, u_rate=u_rate, r=0.0, source=src)


u_sudan = 1.645 * 1 / 2.9          # 1.645 · SD / mean (IPCC 2019 table 4.9: 2.9 ± 1 t d.m./ha/yr)
removal_rates = pd.DataFrame([
    growth('Secondary forest', ZO,  3.29,  .34,     'FREL 2024'),
    growth('Secondary forest', ZM,  2.35,  .34,     'FREL 2024'),
    growth('Secondary forest', ZS,  1.363, u_sudan, 'FREL 2024 (2.9 t d.m. × 0.47)'),
    growth('Secondary forest', ZSS, 1.363, u_sudan, 'Value of the Sudanian zone (same FAO ecological zone)'),
    growth('Agroforest', ZO,  3.16, .75, 'FREL 2024'),
    growth('Agroforest', ZM,  3.16, .75, 'Value of the other zones'),
    growth('Agroforest', ZS,  3.16, .75, 'Value of the other zones'),
    growth('Agroforest', ZSS, 3.16, .75, 'FREL 2024'),
    growth('Plantation / reforestation', ZO,  4.23, .67, 'Value of the mesophilic zone'),
    growth('Plantation / reforestation', ZM,  4.23, .67, 'FREL 2024'),
    growth('Plantation / reforestation', ZS,  4.23, .67, 'Value of the mesophilic zone'),
    growth('Plantation / reforestation', ZSS, 4.23, .67, 'Value of the mesophilic zone'),
])
removal_rates

## 8. Mapped stratum areas by zone

Same values as in notebook `2` (GEE script: https://code.earthengine.google.com/6d246c730ae5efc836ced21ef90d74ac).

In [ ]:
area_hz = pd.DataFrame({
    'Secteur Ombrophile':    [28127.4082,	21377.56608,	8586.479557],
        'Secteur Soudanais':     [86493.21318,	39087.48208,	2729.667459],
        'Secteur Sub Soudanais': [21759.34011,	13883.2849,	1990.485385],
        'Secteur mésophile':     [54425.51251,	34889.54318,	7069.012614],
}, index=pd.Index([1, 2, 3], name='stratum')).T * 100      # km² -> ha
area_hz.index.name = 'zone'
area_hz = area_hz.loc[sorted(area_hz.index)]
area_hz

## 9. Producing the workbook

In [ ]:
from sbae_emissions_excel_en import build_emissions_workbook

samples = valid[['plotid', 'strata', 'zone', 'activity', 'class_initial', 'class_final', 'def_year',
                 'center_lon', 'center_lat']].reset_index(drop=True)
os.makedirs(os.path.dirname(xlsx_path), exist_ok=True)
xl = build_emissions_workbook(
    xlsx_path, samples, area_hz,
    cats={'forest': forest_cats, 'post': post_cats, 'gain': gain_cats},
    maps={'forest': forest_map, 'post': post_map, 'gain': gain_map},
    forest_stocks=forest_stocks, post_stocks=post_stocks, soc_factors=soc_factors, removal_rates=removal_rates,
    confidence=confidence, ef_confidence=ef_confidence, initial_year=initial_year, final_year=final_year,
    socref=socref, u_socref=u_socref, soc_d=soc_d, fire=fire, fire_fraction=fire_fraction,
    sources={'data': os.path.basename(data_path),
             'areas': 'GEE script https://code.earthengine.google.com/6d246c730ae5efc836ced21ef90d74ac',
             'mapping_notes': mapping_notes},
    extra_cols=[('Event year', 'def_year'), ('Longitude', 'center_lon'), ('Latitude', 'center_lat')])
print('Workbook written:', os.path.abspath(xlsx_path))

## 10. Independent checks

1. The areas of the workbook are identical to those of the stratified estimator of notebook `2`, applied transition by transition.
2. The emissions and removals of each component are recomputed here directly from the areas and the factors.

In [ ]:
# 1. Areas: independent stratified estimator (same function as in notebook 2)
def stratified_area_estimate(df, indicator, weights, total_area, z):
    n_h = df.groupby('strata').size()
    n_hk = df[indicator].groupby(df['strata']).sum().reindex(n_h.index, fill_value=0)
    p_h = n_hk / n_h
    w = weights.reindex(n_h.index)
    p = (w * p_h).sum()
    se = np.sqrt((w ** 2 * p_h * (1 - p_h) / (n_h - 1)).sum()) * total_area
    return p * total_area, se


z = round(NormalDist().inv_cdf(1 - (1 - confidence) / 2), 3)
T = final_year - initial_year


def check_areas(rows, code, condition):
    for _, r in rows.iterrows():
        dz = samples[samples.zone == r.zone].copy()
        dz['k'] = ((dz.activity == code) & condition(dz, r)).astype(int)
        a_h = area_hz.loc[r.zone]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, r.area) and np.isclose(se, r.se), (code, r.key)


check_areas(xl.def_rows, 'Def', lambda d, r: (d.class_initial.map(forest_map) == r.forest)
            & (d.class_final.map(post_map) == r.post))
check_areas(xl.deg_rows, 'Deg', lambda d, r: (d.class_initial.map(forest_map) == r.f_i)
            & (d.class_final.map(forest_map) == r.f_f))
check_areas(xl.gain_rows, 'Gain', lambda d, r: d.class_final.map(gain_map) == r.gain)
for code in ['Def', 'Deg', 'Gain']:
    zd = xl.zone_act[code].set_index('zone')
    for zn in area_hz.index:
        dz = samples[samples.zone == zn].assign(k=lambda d: (d.activity == code).astype(int))
        a_h = area_hz.loc[zn]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, zd.loc[zn, 'area']) and np.isclose(se, zd.loc[zn, 'se'])
print('Areas by transition and by zone (Def, Deg, Gain): identical to the independent estimator.')

In [ ]:
# 2. Emissions and removals recomputed directly
fs = {k: v['ef'] for k, v in xl.forest_stocks.items()}
ps = {k: v['ef'] for k, v in xl.post_stocks.items()}
F = {r.category: r.flu * r.fmg * r.fi for r in soc_factors.itertuples()}
G = {(r.category, r.zone): -r.rate * (1 + r.r) * 44 / 12 for r in removal_rates.itertuples()}
cohorts = (T + 1) / 2

d = xl.def_rows
assert np.allclose(d.em_net, d.area * (d.apply(lambda r: fs[(r.forest, r.zone)], axis=1) - d.post.map(ps)))
s = xl.soc_rows
assert np.allclose(s.em, s.area * (-socref * (s.post.map(F) - 1) / soc_d * 44 / 12) * cohorts)
# burning: A · burnt fraction · MB · Cf · Gef · 10⁻³ · GWP, with MB = AGB + litter / 0.37 + deadwood
mb = {k: r.agb + v['litter'] / fire['cf_litter'] + v['dw']
      for k, v in xl.forest_stocks.items() for r in [forest_stocks.set_index(['category', 'zone']).loc[k]]}
burnt = d.area * d.apply(lambda r: fire_fraction[(r.forest, r.post)] * mb[(r.forest, r.zone)], axis=1) * fire['comb']
for gas in ['ch4', 'n2o']:
    assert np.allclose(xl.fire_rows[f'em_{gas}'], burnt * fire[f'gef_{gas}'] * 1e-3 * fire[f'gwp_{gas}'])
g = xl.deg_rows
assert np.allclose(g.em, g.area * g.apply(lambda r: fs[(r.f_i, r.zone)] - fs[(r.f_f, r.zone)], axis=1))
a = xl.gain_rows
assert np.allclose(a.em, a.area * a.apply(lambda r: G[(r.gain, r.zone)], axis=1) * cohorts)
assert np.allclose(xl.direct.set_index('zone').em_net, d.groupby('zone').em_net.sum())
assert np.isclose(xl.years[['def', 'soc', 'ch4', 'n2o', 'deg', 'enh']].sum().sum(), xl.balance['net']['period'])
print('DEF emissions (biomass, soils, burning CH₄ and N₂O), degradation and removals: identical to the direct '
      'calculation; yearly series consistent with the period total.')

## 11. Results

Annual averages over the period; 90% uncertainties expressed in % of the value and in absolute value (±). Removals are negative.

In [ ]:
names = {'def': 'Deforestation - biomass and DOM (net EF)', 'soc': 'Deforestation - mineral soils',
         'ch4': 'Deforestation - biomass burning, CH4', 'n2o': 'Deforestation - biomass burning, N2O',
         'deg': 'Degradation', 'tot': 'Total emissions', 'enh': 'Enhancement of stocks (gains)',
         'net': 'Net balance'}
balance = pd.DataFrame([{'component': names[c], 'tCO2e/yr': v['year'], 'uncertainty': v['u'], '± tCO2e/yr': v['pm'],
                         'period total (tCO2e)': v['period']} for c, v in xl.balance.items() if c in names]).set_index('component')
balance = balance.loc[[names[c] for c in ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'enh', 'net']]]
balance.style.format({'tCO2e/yr': '{:,.0f}', 'uncertainty': '{:.1%}', '± tCO2e/yr': '{:,.0f}',
                      'period total (tCO2e)': '{:,.0f}'})

In [ ]:
print('Annual areas (direct estimate):')
areas = pd.DataFrame([{'activity': a, 'ha/yr': n['area_year'], 'uncertainty': n['u_area'], '± ha/yr': n['u_area'] * n['area_year']}
                      for a, n in ((a, xl.zone_act[a].attrs['national']) for a in ['Def', 'Deg', 'Gain'])]).set_index('activity')
areas.style.format({'ha/yr': '{:,.0f}', 'uncertainty': '{:.1%}', '± ha/yr': '{:,.0f}'})

In [ ]:
print('Balance by zone (tCO2e/yr):')
components = ['def', 'soc', 'ch4', 'n2o', 'deg', 'enh', 'net']
rows = []
for zn, comp in list(xl.balance_zone.items()) + [('National', xl.balance)]:
    row = {'zone': zn}
    for c in components:
        row[c] = comp[c]['year']
        row[f'U {c}'] = comp[c]['u']
    rows.append(row)
by_zone = pd.DataFrame(rows).set_index('zone')
by_zone.style.format({c: '{:,.0f}' for c in components} | {f'U {c}': '{:.1%}' for c in components})

In [ ]:
print("Yearly series (soils and removals accumulate from one year to the next):")
cols = ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'enh', 'net', 'pm_net']
series = xl.years.set_index('year')[cols]
series['U net'] = series.pm_net / series.net.abs()
series.style.format({c: '{:,.0f}' for c in cols} | {'U net': '{:.1%}'})

## 12. Estimates for the ERP

Same calculation chain, applied to the samples located in the ERP only (column `pre` produced by notebook `1`), with the mapped stratum areas **within the ERP**, by zone. Factors, mappings and parameters are those of sections 4 to 7. The ERP total is the sum of its zones.

The ERP does not overlap the Secteur Soudanais. A zone of the ERP without any sample cannot be estimated: it is removed (reported below).

In [ ]:
xlsx_path_erp = 'results/CIV_emissions_deforestation_2021_2025_ERP_en.xlsx'

# Mapped area of each stratum within the ERP, by zone (km² -> ha)
area_hz_erp = pd.DataFrame({
    'Secteur Ombrophile':    [17262.20123016427,  12174.209957087007, 4988.553467034733],
    'Secteur Sub Soudanais': [24.385942521542056, 9.404869314886117,  0.24926422661324296],
    'Secteur mésophile':     [6623.2923265831605, 4158.777891905002,  710.699415400892],
}, index=pd.Index([1, 2, 3], name='stratum')).T * 100
area_hz_erp.index.name = 'zone'
area_hz_erp = area_hz_erp.loc[sorted(area_hz_erp.index)]

samples_erp = samples[samples.plotid.isin(valid.loc[valid['pre'], 'plotid'])].reset_index(drop=True)
assert ZS not in set(samples_erp.zone), 'ERP sample assigned to the Secteur Soudanais'
print(len(samples_erp), 'valid samples in the ERP')

zones_without_samples = [zn for zn in area_hz_erp.index if zn not in set(samples_erp.zone)]
for zn in zones_without_samples:
    print(f"{zn}: no sample for {area_hz_erp.loc[zn].sum():,.0f} ha "
          f"({area_hz_erp.loc[zn].sum() / area_hz_erp.to_numpy().sum():.2%} of the ERP) -> zone removed")
area_hz_erp = area_hz_erp.drop(index=zones_without_samples)
pd.crosstab(samples_erp.activity, samples_erp.zone, margins=True)

In [ ]:
xl_erp = build_emissions_workbook(
    xlsx_path_erp, samples_erp, area_hz_erp,
    cats={'forest': forest_cats, 'post': post_cats, 'gain': gain_cats},
    maps={'forest': forest_map, 'post': post_map, 'gain': gain_map},
    forest_stocks=forest_stocks, post_stocks=post_stocks, soc_factors=soc_factors, removal_rates=removal_rates,
    confidence=confidence, ef_confidence=ef_confidence, initial_year=initial_year, final_year=final_year,
    socref=socref, u_socref=u_socref, soc_d=soc_d, fire=fire, fire_fraction=fire_fraction,
    title="Côte d'Ivoire — ERP — Forest emissions and removals", territory='ERP',
    sources={'data': os.path.basename(data_path) + ' (samples located in the ERP)',
             'areas': 'stratum areas within the ERP, by zone (provided)'
                      + (f" ; zone without samples removed: {', '.join(zones_without_samples)}" if zones_without_samples else ''),
             'mapping_notes': mapping_notes},
    extra_cols=[('Event year', 'def_year'), ('Longitude', 'center_lon'), ('Latitude', 'center_lat')])
print('Workbook written:', os.path.abspath(xlsx_path_erp))

In [ ]:
# Check: areas by zone and ERP emissions vs independent calculation
for code in ['Def', 'Deg', 'Gain']:
    zd = xl_erp.zone_act[code].set_index('zone')
    for zn in area_hz_erp.index:
        dz = samples_erp[samples_erp.zone == zn].assign(k=lambda d: (d.activity == code).astype(int))
        a_h = area_hz_erp.loc[zn]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, zd.loc[zn, 'area']) and np.isclose(se, zd.loc[zn, 'se'])
d = xl_erp.def_rows
assert np.allclose(d.em_net, d.area * (d.apply(lambda r: fs[(r.forest, r.zone)], axis=1) - d.post.map(ps)))
g = xl_erp.deg_rows
assert np.allclose(g.em, g.area * g.apply(lambda r: fs[(r.f_i, r.zone)] - fs[(r.f_f, r.zone)], axis=1))
a = xl_erp.gain_rows
assert np.allclose(a.em, a.area * a.apply(lambda r: G[(r.gain, r.zone)], axis=1) * cohorts)
assert np.isclose(xl_erp.years[['def', 'soc', 'ch4', 'n2o', 'deg', 'enh']].sum().sum(), xl_erp.balance['net']['period'])
print('ERP: areas (Def, Deg, Gain) and DEF, DEG and gain emissions identical to the independent calculation.')

In [ ]:
print('ERP: annual averages over the period (tCO2e/yr):')
balance_erp = pd.DataFrame([{'component': names[c], 'tCO2e/yr': v['year'], 'uncertainty': v['u'], '± tCO2e/yr': v['pm'],
                             'period total (tCO2e)': v['period']} for c, v in xl_erp.balance.items() if c in names]).set_index('component')
balance_erp = balance_erp.loc[[names[c] for c in ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'enh', 'net']]]
balance_erp.style.format({'tCO2e/yr': '{:,.0f}', 'uncertainty': '{:.1%}', '± tCO2e/yr': '{:,.0f}',
                          'period total (tCO2e)': '{:,.0f}'})

In [ ]:
print('ERP: annual areas (direct estimate):')
areas_erp = pd.DataFrame([{'activity': a, 'ha/yr': n['area_year'], 'uncertainty': n['u_area'], '± ha/yr': n['u_area'] * n['area_year']}
                          for a, n in ((a, xl_erp.zone_act[a].attrs['national']) for a in ['Def', 'Deg', 'Gain'])]).set_index('activity')
areas_erp.style.format({'ha/yr': '{:,.0f}', 'uncertainty': '{:.1%}', '± ha/yr': '{:,.0f}'})

In [ ]:
print('ERP: balance by zone (tCO2e/yr):')
rows = []
for zn, comp in list(xl_erp.balance_zone.items()) + [('ERP', xl_erp.balance)]:
    row = {'zone': zn}
    for c in components:
        row[c] = comp[c]['year']
        row[f'U {c}'] = comp[c]['u']
    rows.append(row)
by_zone_erp = pd.DataFrame(rows).set_index('zone')
by_zone_erp.style.format({c: '{:,.0f}' for c in components} | {f'U {c}': '{:.1%}' for c in components})